# Nonlinear Cart-Pole control practice

비선형 Cart-Pole 제어계 하나를 기준으로 수치해석, PINN, Neural ODE, 강화학습을 비교합니다.

상태공간은 $X=\mathbb{R}^4$이고 상태는

$$
x(t)=(p(t),\theta(t),\dot p(t),\dot\theta(t))\in X
$$

입니다. 여기서 $p$는 cart 위치, $\theta$는 위쪽 평형점에서의 pole 각도입니다. 제어입력은 force $u(t)\in\mathbb{R}$입니다.

비선형 운동방정식은 하나의 map

$$
f:X\times\mathbb{R}\to X,
$$

$$
\dot x(t)=f(x(t),u(t))
$$

로 둡니다.

목적은 같은 동역학을 네 가지 관점으로 보는 것입니다.

1. DOP853 + LQR: 고정밀 numerical reference와 고전 제어 baseline
2. PINN: 알려진 closed-loop 운동방정식 자체를 만족하는 trajectory 학습
3. Neural ODE: solver가 만든 trajectory에서 비선형 vector field 학습
4. RL: 운동방정식을 직접 맞추지 않고 PPO로 제어정책 학습


In [1]:
!git clone -q https://github.com/HisameOgasahara/RL_tmp.git
%cd RL_tmp
!pip install -q -r requirements.txt
import sys
sys.path.append("src")


/content/RL_tmp


## 1. 공통 설정

$T>0$는 최종 시간, $\Delta t>0$는 시간 간격입니다. 초기 상태는 $x_0=(p_0,\theta_0,\dot p_0,\dot\theta_0)$입니다.

이번 예제는 upright 근처의 초기 상태 $\theta_0=0.2\,\mathrm{rad}$에서 시작합니다.


In [2]:
import numpy as np
import torch

from rl_tmp.cartpole_system import CartPoleConfig
from rl_tmp.cartpole_solver import (
    make_lqr_controller,
    solve_cartpole_reference,
)
from rl_tmp.cartpole_visualize import (
    plot_state_trajectories,
    plot_training,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

t_final = 5.0
dt = 0.02

initial_state = np.array(
    [0.0, 0.2, 0.0, 0.0],
    dtype=np.float64,
)

config = CartPoleConfig()
times = np.arange(0.0, t_final + 1e-9, dt)

print("device:", device)
print("time steps:", len(times))
print("initial state:", initial_state)


device: cpu
time steps: 251
initial state: [0.  0.2 0.  0. ]


## 2. DOP853 reference + LQR

먼저 비선형 ODE를 SciPy의 adaptive high-order solver DOP853로 적분합니다. 이 궤적을 이후 방법들의 reference로 사용합니다.

고전 제어 baseline은 평형점 $x=0$에서 비선형 map $f$를 선형화한

$$
\dot x(t)=Ax(t)+Bu(t)
$$

에 대해 LQR을 사용합니다. 상태비용 행렬 $Q\in\mathbb{R}^{4\times4}$와 제어비용 $R\in\mathbb{R}^{1\times1}$에 대해 Riccati equation을 풀어 feedback map

$$
\pi_{\mathrm{LQR}}:\mathbb{R}^4\to\mathbb{R},\qquad
\pi_{\mathrm{LQR}}(x)=-Kx
$$

를 얻습니다. LQR은 선형화된 계의 잘 알려진 baseline이므로 RL 결과가 실제로 제어를 배우는지 비교하는 데 사용합니다.


In [3]:
lqr_policy, lqr_gain, A, B = make_lqr_controller(config)

uncontrolled = solve_cartpole_reference(
    initial_state=initial_state,
    times=times,
    config=config,
)

lqr_trajectory = solve_cartpole_reference(
    initial_state=initial_state,
    times=times,
    config=config,
    control_law=lqr_policy,
)

print("LQR gain K:")
print(lqr_gain)
print(
    "uncontrolled final state norm:",
    np.linalg.norm(uncontrolled[-1]),
)
print(
    "LQR final state norm:",
    np.linalg.norm(lqr_trajectory[-1]),
)

plot_state_trajectories(
    times=times,
    reference=uncontrolled,
    prediction=lqr_trajectory,
    labels=("uncontrolled DOP853", "LQR + DOP853"),
    title="Cart-Pole: uncontrolled vs LQR",
)


LQR gain K:
[[ -2.5819889  -45.8462856   -5.0144165  -12.09511023]]
uncontrolled final state norm: 6.768948823459855
LQR final state norm: 0.019255141785996098


## 3. PINN

PINN에서는 LQR을 넣은 비선형 closed-loop ODE

$$
\dot x(t)=f\bigl(x(t),\pi_{\mathrm{LQR}}(x(t))\bigr)
$$

를 직접 만족하도록 trajectory map

$$
x_\theta:[0,T]\to\mathbb{R}^4
$$

를 학습합니다.

초기 실험에서는 초기조건 loss와 physics residual을 따로 최적화했을 때 trajectory 오차가 크게 남았습니다. 그래서 현재 구현은

$$
x_\theta(t)=x_0+\frac{t}{T}N_\theta\!\left(\frac{t}{T}\right)
$$

로 parameterization하여 $x_\theta(0)=x_0$를 구조적으로 보장합니다.


In [4]:
from rl_tmp.cartpole_pinn import (
    predict_cartpole_pinn,
    train_cartpole_pinn,
)

pinn_t_final = 1.0
pinn_times = np.linspace(0.0, pinn_t_final, 61)

pinn_reference = solve_cartpole_reference(
    initial_state=initial_state,
    times=pinn_times,
    config=config,
    control_law=lqr_policy,
)

pinn_result = train_cartpole_pinn(
    initial_state=initial_state,
    config=config,
    lqr_gain=lqr_gain,
    t_final=pinn_t_final,
    steps=3000,
    collocation_points=160,
    seed=0,
    device=device,
)

pinn_prediction = predict_cartpole_pinn(
    model=pinn_result.model,
    times=pinn_times,
    device=device,
).numpy()

pinn_mse = np.mean(
    (pinn_prediction - pinn_reference) ** 2
)

pinn_final_error = np.linalg.norm(
    pinn_prediction[-1] - pinn_reference[-1]
)

print(
    "PINN residual:",
    pinn_result.history["residual"][0],
    "->",
    pinn_result.history["residual"][-1],
)
print("trajectory MSE:", pinn_mse)
print("final-state error:", pinn_final_error)

plot_training(
    pinn_result.history,
    title="Cart-Pole PINN residual",
)

plot_state_trajectories(
    times=pinn_times,
    reference=pinn_reference,
    prediction=pinn_prediction,
    labels=("LQR + DOP853", "PINN"),
    title="PINN closed-loop trajectory",
)


PINN residual: 45.664283752441406 -> 0.09164108335971832
trajectory MSE: 0.003541154409102019
final-state error: 0.0976341237650926


## 4. Neural ODE

Neural ODE에서는 운동방정식을 알려주지 않고 DOP853가 만든 여러 trajectory에서 vector field

$$
f_\theta:\mathbb{R}^4\to\mathbb{R}^4
$$

를 학습합니다.

즉 PINN은 알려진 물리식의 residual을 최소화하고, Neural ODE는 solver data에서 미지의 dynamics를 근사한다는 차이가 있습니다. 학습 뒤에는 $\dot x=f_\theta(x)$를 다시 적분하여 보지 않은 초기조건의 reference trajectory와 비교합니다.


In [5]:
from rl_tmp.cartpole_neural_ode import (
    rollout_learned_field,
    train_cartpole_neural_ode,
)

neural_result = train_cartpole_neural_ode(
    config=config,
    steps=800,
    seed=0,
    device=device,
)

neural_times = np.linspace(0.0, 1.0, 41)
neural_initial_state = np.array(
    [0.1, 0.12, 0.0, 0.0],
    dtype=np.float64,
)

neural_reference = solve_cartpole_reference(
    initial_state=neural_initial_state,
    times=neural_times,
    config=config,
)

neural_prediction = rollout_learned_field(
    model=neural_result.model,
    initial_state=neural_initial_state,
    times=neural_times,
    device=device,
).numpy()

neural_mse = np.mean(
    (neural_prediction - neural_reference) ** 2
)

neural_final_error = np.linalg.norm(
    neural_prediction[-1] - neural_reference[-1]
)

print(
    "field MSE:",
    neural_result.history["field_mse"][0],
    "->",
    neural_result.history["field_mse"][-1],
)
print("trajectory MSE:", neural_mse)
print("final-state error:", neural_final_error)

plot_training(
    neural_result.history,
    title="Neural ODE vector-field training",
)

plot_state_trajectories(
    times=neural_times,
    reference=neural_reference,
    prediction=neural_prediction,
    labels=("DOP853", "Neural ODE"),
    title="Neural ODE rollout",
)


field MSE: 28.405162811279297 -> 0.006900671869516373
trajectory MSE: 0.0051822790842860275
final-state error: 0.22482979201546854


## 5. RL control

RL에서는 상태 $x_t\in\mathbb{R}^4$를 받아 두 force 중 하나의 확률분포를 만드는 정책

$$
\pi_\theta:\mathbb{R}^4\to\mathcal{P}(\{-10,+10\})
$$

를 PPO로 학습합니다.

목적은 pole만 순간적으로 세우는 것이 아니라 cart가 $|p|\le2.4$ 안에 있으면서 $|\theta|\le0.5$를 유지하는 것입니다. 초기 reward에서는 pole angle에 비해 cart 위치 penalty가 약해서 deterministic rollout이 cart 경계 밖으로 drift했습니다. 그래서 cart 위치·속도와 failure penalty를 키운 뒤 다시 학습했습니다.


In [6]:
from rl_tmp.cartpole_rl import (
    rollout_cartpole_policy,
    train_cartpole_ppo,
)

rl_result = train_cartpole_ppo(
    config=config,
    updates=60,
    batch_size=64,
    horizon=96,
    dt=dt,
    seed=0,
    device=device,
)

controlled_states, actions, success = rollout_cartpole_policy(
    model=rl_result.model,
    initial_state=initial_state,
    config=config,
    t_final=t_final,
    dt=dt,
    device=device,
)

print(
    "failure rate:",
    rl_result.history["failure_rate"][0],
    "->",
    rl_result.history["failure_rate"][-1],
)
print("5 s rollout success:", success)
print(
    "max |p|:",
    np.max(np.abs(controlled_states[:, 0])),
)
print(
    "max |theta|:",
    np.max(np.abs(controlled_states[:, 1])),
)
print("final state:", controlled_states[-1])

plot_training(
    rl_result.history,
    title="Cart-Pole PPO training",
)

rollout_times = np.arange(
    len(controlled_states)
) * dt

plot_state_trajectories(
    times=rollout_times,
    reference=lqr_trajectory[:len(controlled_states)],
    prediction=controlled_states,
    labels=("LQR + DOP853", "PPO"),
    title="LQR baseline vs PPO rollout",
)


failure rate: 0.0338541679084301 -> 0.009765625
5 s rollout success: True
max |p|: 0.4804271
max |theta|: 0.2
final state: [ 0.12194668 -0.00414611  0.00157434 -0.03463274]


## 결과를 볼 때

- DOP853 + LQR: 비선형 solver reference와 고전 제어 baseline이 upright로 수렴하는지 확인
- PINN: 알려진 closed-loop ODE residual을 줄인 trajectory가 DOP853 reference를 따라가는지 확인
- Neural ODE: solver data에서 학습한 vector field가 보지 않은 초기조건의 trajectory를 재현하는지 확인
- RL: 운동방정식 residual 없이 학습한 정책이 cart와 pole의 제한을 지키며 실제 환경 rollout을 수행하는지 확인
